# 09 — Modèles de boosting avancés

## Objectif du notebook

Cette étape étudie deux implémentations modernes du Gradient Boosting :

- **XGBoost** ;
- **LightGBM**.

L’objectif est de déterminer si leurs mécanismes d’optimisation, de régularisation et de construction des arbres permettent d’exploiter plus efficacement le signal contenu dans les données que le Gradient Boosting classique étudié précédemment.

Les modèles sont évalués selon le même protocole expérimental que les étapes antérieures :

- cible binaire identique ;
- validation temporelle en expanding window ;
- mêmes quatre folds ;
- aucune séparation aléatoire ;
- mêmes métriques : Accuracy, ROC-AUC et Log-Loss ;
- aucune utilisation du jeu de test officiel pour sélectionner les modèles.

Les expériences de ce notebook utilisent l’ensemble des variables numériques disponibles après feature engineering, à l’exception des identifiants, de la cible et de la variable `ALLOCATION`.

## Pourquoi étudier XGBoost et LightGBM ?

Le Gradient Boosting classique a produit la meilleure performance publique obtenue jusqu’à présent dans le projet. Cela suggère que les relations entre les rendements historiques et la cible contiennent des effets non linéaires, des seuils ou des interactions que la régression logistique ne parvient pas à représenter.

XGBoost et LightGBM reposent sur le même principe additif que le Gradient Boosting, mais introduisent des optimisations supplémentaires.

### XGBoost

XGBoost améliore le Gradient Boosting grâce à :

- une fonction objectif régularisée ;
- l’utilisation des gradients et des Hessiennes ;
- une pénalisation de la complexité des arbres ;
- la régularisation L1 et L2 des valeurs des feuilles ;
- le sous-échantillonnage des lignes et des variables.

### LightGBM

LightGBM se distingue notamment par :

- une recherche des seuils fondée sur des histogrammes ;
- une croissance des arbres feuille par feuille ;
- une sélection prioritaire de la feuille offrant le plus grand gain ;
- une exécution généralement plus rapide et moins coûteuse en mémoire.

Ces optimisations ne garantissent cependant pas automatiquement de meilleures performances prédictives. Leur intérêt doit être vérifié dans le cadre de la validation temporelle du projet.

In [1]:
import sys
from pathlib import Path

ROOT = Path.cwd().parent

if str(ROOT) not in sys.path:
    sys.path.append(str(ROOT))

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from src.data_loading import load_X_train, load_y_train
from src.validation import create_expanding_window_folds, check_temporal_folds
from src.evaluation import evaluate_model_on_folds, compare_model_results
from src.target import create_class_column
from src.advanced_boosting import (
    build_xgboost_pipeline, 
    build_lightgbm_pipeline 
)
from src.features import (
    create_momentum_features, 
    create_volatility_features, 
    create_momentum_volatility_ratio_features,
    create_volume_features
)

In [3]:
X_train = load_X_train()
y_train = load_y_train()

In [4]:
df_train, _ = create_class_column(X_train, y_train)

In [5]:
df_train = create_momentum_features(df_train)
df_train = create_volatility_features(df_train)
df_train = create_momentum_volatility_ratio_features(df_train)
df_train = create_volume_features(df_train)

In [6]:
dates = sorted(list(set(X_train["TS"])))
folds = create_expanding_window_folds(dates)
check_temporal_folds(folds,dates,validation_size=120)

True

In [7]:
drop_cols = ['ROW_ID', 'TS', 'ALLOCATION', 'target', 'class']

In [8]:
ret_features = [
    'RET_20', 'RET_19', 'RET_18', 'RET_17',
    'RET_16', 'RET_15', 'RET_14', 'RET_13', 
    'RET_12', 'RET_11', 'RET_10','RET_9', 
    'RET_8', 'RET_7', 'RET_6', 'RET_5', 
    'RET_4', 'RET_3', 'RET_2','RET_1',
    'volume_3','volume_20', 'delta_volume_3_20'
]

In [9]:
features = df_train.drop(
    columns = drop_cols
).columns

## xgboost

In [10]:
xgboost = build_xgboost_pipeline()

In [11]:
xgboost_results = evaluate_model_on_folds(df_train,folds,xgboost,features)

In [12]:
xgboost_results

,fold,train_start,train_end,valid_start,valid_end,train_positive_rate,valid_positive_rate,accuracy,log_loss,roc_auc,n_valid_predictions,n_correct_predictions
0,1,DATE_0001,DATE_2042,DATE_2043,DATE_2162,0.504803,0.522228,0.517625,0.691962,0.522467,24114,12482
1,2,DATE_0001,DATE_2162,DATE_2163,DATE_2282,0.505732,0.504837,0.535129,0.690363,0.552960,24396,13055
2,3,DATE_0001,DATE_2282,DATE_2283,DATE_2402,0.505687,0.520554,0.525077,0.691687,0.527631,24764,13003
3,4,DATE_0001,DATE_2402,DATE_2403,DATE_2522,0.506421,0.522097,0.523149,0.691514,0.527154,25660,13424


In [13]:
results_str_cols = [
    'fold', 'train_start', 
    'train_end', 'valid_start', 
    'valid_end'
]

In [14]:
xgboost_results.drop(
    columns=results_str_cols
).mean(axis=0)

train_positive_rate          0.505661
valid_positive_rate          0.517429
accuracy                     0.525245
log_loss                     0.691382
roc_auc                      0.532553
n_valid_predictions      24733.500000
n_correct_predictions    12991.000000
dtype: float64

## lgbm

In [15]:
lgbm = build_lightgbm_pipeline()

In [ ]:
lgbm_results = evaluate_model_on_folds(df_train,folds,lgbm,features)

In [17]:
lgbm_results

,fold,train_start,train_end,valid_start,valid_end,train_positive_rate,valid_positive_rate,accuracy,log_loss,roc_auc,n_valid_predictions,n_correct_predictions
0,1,DATE_0001,DATE_2042,DATE_2043,DATE_2162,0.504803,0.522228,0.519366,0.691996,0.522282,24114,12524
1,2,DATE_0001,DATE_2162,DATE_2163,DATE_2282,0.505732,0.504837,0.535129,0.690452,0.551281,24396,13055
2,3,DATE_0001,DATE_2282,DATE_2283,DATE_2402,0.505687,0.520554,0.522533,0.691780,0.526214,24764,12940
3,4,DATE_0001,DATE_2402,DATE_2403,DATE_2522,0.506421,0.522097,0.524240,0.691555,0.526920,25660,13452


In [18]:
lgbm_results.drop(
    columns=results_str_cols
).mean(axis=0)

train_positive_rate          0.505661
valid_positive_rate          0.517429
accuracy                     0.525317
log_loss                     0.691446
roc_auc                      0.531674
n_valid_predictions      24733.500000
n_correct_predictions    12992.750000
dtype: float64

## 7. Conclusion

Cette étape a permis d’évaluer XGBoost et LightGBM sur l’ensemble des variables numériques disponibles après feature engineering, tout en conservant le protocole strict de validation temporelle du projet.

Les deux modèles obtiennent des résultats très proches :

| Modèle | Accuracy moyenne | ROC-AUC moyen | Log-Loss moyenne |
|---|---:|---:|---:|
| XGBoost | 0,5252 | 0,5326 | 0,6914 |
| LightGBM | 0,5253 | 0,5317 | 0,6914 |

Aucun des deux modèles ne domine clairement l’autre.

Les principaux enseignements sont les suivants :

1. les modèles extraient un signal prédictif faible mais supérieur au hasard ;
2. les performances varient sensiblement selon les périodes ;
3. le deuxième fold est nettement plus favorable que les autres ;
4. l’utilisation de toutes les variables numériques ne produit pas une amélioration spectaculaire ;
5. le changement d’implémentation du boosting ne suffit pas à résoudre la faiblesse du signal ;
6. la limite actuelle semble davantage liée à la représentation du problème et à la quantité de signal disponible qu’à la puissance brute des algorithmes.

À ce stade, XGBoost et LightGBM ne justifient pas une optimisation exhaustive tant qu’ils n’ont pas démontré une supériorité claire face au Gradient Boosting classique.